# Internals do carteira_auto
Exploração das APIs internas do sistema.

## 1. Result Type — Error Handling Explícito

In [ ]:
from carteira_auto.core import Ok, Err

# Sucesso
result = Ok(42)
print(f"is_ok: {result.is_ok()}, value: {result.unwrap()}")

# Erro
err = Err("falha na API", {"status": 500})
print(f"is_ok: {err.is_ok()}, fallback: {err.unwrap_or(0)}")
print(f"error: {err.error}, details: {err.details}")

## 2. Pydantic Models — Validação Estrita

In [ ]:
from carteira_auto.core.models import Asset, Portfolio
from pydantic import ValidationError

# Asset válido
asset = Asset(ticker="PETR4", nome="Petrobras PN", preco_atual=35.0)
print(f"Ticker: {asset.ticker}, Preço: {asset.preco_atual}")

# Validação: ticker vazio falha
try:
    Asset(ticker="", nome="Teste")
except ValidationError as e:
    print(f"Erro esperado: {e.errors()[0]['msg']}")

# Validação: preço negativo falha
try:
    Asset(ticker="VALE3", nome="Vale", preco_atual=-10.0)
except ValidationError as e:
    print(f"Erro esperado: {e.errors()[0]['msg']}")

# model_copy para imutabilidade
updated = asset.model_copy(update={"preco_atual": 36.50})
print(f"Original: {asset.preco_atual}, Cópia: {updated.preco_atual}")

## 3. DAGEngine — Orquestração de Pipelines

In [ ]:
from carteira_auto.core.engine import DAGEngine, Node, PipelineContext

class SomaNode(Node):
    name = "soma"
    dependencies: list[str] = []
    def run(self, ctx):
        ctx["resultado"] = ctx.get("a", 0) + ctx.get("b", 0)
        return ctx

class MultiplicaNode(Node):
    name = "multiplica"
    dependencies = ["soma"]
    def run(self, ctx):
        ctx["final"] = ctx["resultado"] * 2
        return ctx

engine = DAGEngine()
engine.register(SomaNode())
engine.register(MultiplicaNode())

# Dry run
print("Plano:", engine.dry_run("multiplica"))

# Execução
ctx = PipelineContext(a=3, b=7)
ctx = engine.run("multiplica", ctx)
print(f"Soma: {ctx['resultado']}, Final: {ctx['final']}")

## 4. Error Handling no DAGEngine

In [ ]:
class FalhaNode(Node):
    name = "falha"
    dependencies: list[str] = []
    def run(self, ctx):
        raise ValueError("Erro simulado")

class SucessoNode(Node):
    name = "sucesso"
    dependencies: list[str] = []
    def run(self, ctx):
        ctx["ok"] = True
        return ctx

# Modo tolerante (fail_fast=False)
engine = DAGEngine(fail_fast=False)
engine.register(FalhaNode())
engine.register(SucessoNode())

class FinalNode(Node):
    name = "final"
    dependencies = ["falha", "sucesso"]
    def run(self, ctx):
        return ctx

engine.register(FinalNode())
ctx = engine.run("final")
print(f"Erros: {ctx.errors}")
print(f"Sucesso executou: {ctx.get('ok')}")
print(f"Tem erros: {ctx.has_errors}")

## 5. PipelineContext — Comunicação entre Nodes

In [ ]:
ctx = PipelineContext()
ctx["portfolio"] = "dados..."
ctx["_errors"] = {"node_x": "falhou"}

print(f"has_errors: {ctx.has_errors}")
print(f"errors: {ctx.errors}")
print(f"get_typed: {ctx.get_typed('portfolio', str)}")

## 6. Registry — Pipelines Registradas

In [ ]:
from carteira_auto.core.registry import list_pipelines, create_engine

# Listar pipelines
for name, desc in sorted(list_pipelines().items()):
    print(f"  {name:35s} {desc}")

# Criar engine e dry_run
engine = create_engine()
plan = engine.dry_run("analyze_portfolio")
print(f"\nPlano para 'analyze': {plan}")